In [ ]:
# import importlib
# import subprocess
# import sys

# def install_if_missing(package_name, import_name=None):
#     try:
#         importlib.import_module(import_name or package_name)
#         print(f"✅ '{package_name}' is already installed.")
#     except ImportError:
#         print(f"📦 Installing '{package_name}'...")
#         subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
#         print(f"✅ '{package_name}' installed successfully.")

# # Example usage
# install_if_missing("pyspark")
# install_if_missing("gspread")
# install_if_missing("oauth2client")
# install_if_missing("matplotlib")
# install_if_missing("seaborn")
# install_if_missing("scikit-learn")
# install_if_missing("pandas")
# install_if_missing("numpy")

In [17]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, avg, sum as spark_sum
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import Correlation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error, r2_score
# importlib.reload(sys.modules[__name__])
# --- IGNORE ---



# --- Configuration ---
ICEBERG_VERSION = "1.5.0"
LOCAL_WAREHOUSE_PATH = "/C:/data/data_files/iceberg/iceberg_warehouse"
CATALOG_NAME = "local"

# --- Stop existing SparkSession ---
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception:
    pass

# --- Clean warehouse ---
if os.path.exists(LOCAL_WAREHOUSE_PATH):
    print(f"Cleaning up old warehouse: {f"file://{LOCAL_WAREHOUSE_PATH}"}")
    shutil.rmtree(f"file://{LOCAL_WAREHOUSE_PATH}")

# --- Iceberg packages ---
ICEBERG_PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
    f"org.apache.avro:avro:1.11.3"
)

## --- SparkSession ---
spark = SparkSession.builder \
    .appName("IcebergDescribeExample") \
    .config("spark.jars.packages", ICEBERG_PACKAGES) \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file://{LOCAL_WAREHOUSE_PATH}") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

# spark = SparkSession.builder \
#     .appName("IcebergDescribeExample") \
#     .config("spark.jars.packages", ICEBERG_PACKAGES) \
#     .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
#     .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop") \
#     .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file://{LOCAL_WAREHOUSE_PATH}") \
#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
#     .getOrCreate()

print("Spark version:", spark.version)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
df = spark.sql("Select 'A' col1")
df.show()
df.printSchema()
df = None

In [ ]:
import urllib.request
# This script fetches data from a specified URL and prints the content.
# Download the all sheets information locally
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQO7atdyPm1anKXSql2oz74C3To18tkzYRIPqhm9YqWX1w3nm73sL8lpybaykBO2g7IB7cvIpQ8W9_u/pub?gid=0&single=true&output=csv"
allsheetsInformation = "AllSheets_Link.csv"

# First, retrieve the data and save it to the local file.
urllib.request.urlretrieve(url, allsheetsInformation)

# Now, open the local file and print its content.
with open(allsheetsInformation, 'r', encoding='utf-8') as response:
    content = response.read()
    
print(content)

In [ ]:
import csv

with open(allsheetsInformation, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    sheet_info_list = list(reader)

# Example: print each sheet's DataFrame name and link
for sheet in sheet_info_list:
    print(f"{sheet['Data_Frame_Name']} → {sheet['Link']}")


In [ ]:
metadata_path = "AllSheets_Link.csv"
metadata_df = spark.read \
    .option("header", True)\
    .option("nullValue", "NULL") \
    .csv(metadata_path)

metadata_df.show()

sheet_info = metadata_df.select("Data_Frame_Name", "Link").collect()
dataframes = {}

for row in sheet_info:
    df_name = row["Data_Frame_Name"]
    csv_url = row["Link"]
    local_path = f"C:/data/data_files/tmp/{df_name}.csv"  # or any writable path

    urllib.request.urlretrieve(csv_url, local_path)

    df = spark.read \
        .option("header", True) \
        .option("nullValue", "NULL") \
        .option("inferSchema", True) \
        .csv(local_path)

    dataframes[df_name] = df
   

In [11]:
# Load Data using saved CSV files.

LOCAL_ICEBERG_CATALOG = 'local'
folder_path = "C:/data/data_files/tmp/" # Replace with your actual folder path
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
dataframes = {}
for file_name in csv_files:
    print(file_name + ' - ' + file_name[:file_name.find("_")] + ' - '  + file_name[file_name.find("_") + 1:-4])
    file_path = os.path.join(folder_path, file_name)
    print("filepath: " + file_path)
    file_name_noext = os.path.splitext(file_name)[0] # Get filename without extension
    print(file_name_noext)

    # # Read the CSV file into a DataFrame
    # # Adjust options like header and inferSchema as needed
    df_csv = spark.read.csv(file_path, header=True, inferSchema=True)

    # This will be the database name (e.g., 'mydb')
    db_name = file_name[:file_name.find("_")] 
    
    # This will be the table name (e.g., 'mytable')
    table_name = file_name[file_name.find("_") + 1:-4]
    
    # 2. Construct the Fully Qualified Iceberg Table Identifier
    # Iceberg requires a three-part name: catalog.database.table
    fq_iceberg_table = f"{LOCAL_ICEBERG_CATALOG}.{db_name}.{table_name}"
    
    print(f"Creating Iceberg table: {fq_iceberg_table}")

    # 3. Create the Iceberg table and write data using the DataFrameWriter API
    df_csv \
        .write \
        .format("iceberg") \
        .mode("overwrite") \
        .saveAsTable(fq_iceberg_table)
    
    # 4. Read the data back from the new Iceberg table
    iceberg_df = spark.read.table(fq_iceberg_table)

    


HumanResources_Department.csv - HumanResources - Department
filepath: C:/data/data_files/tmp/HumanResources_Department.csv
HumanResources_Department
Creating Iceberg table: local.HumanResources.Department
HumanResources_Employee.csv - HumanResources - Employee
filepath: C:/data/data_files/tmp/HumanResources_Employee.csv
HumanResources_Employee
Creating Iceberg table: local.HumanResources.Employee
HumanResources_EmployeeDepartmentHistory.csv - HumanResources - EmployeeDepartmentHistory
filepath: C:/data/data_files/tmp/HumanResources_EmployeeDepartmentHistory.csv
HumanResources_EmployeeDepartmentHistory
Creating Iceberg table: local.HumanResources.EmployeeDepartmentHistory
HumanResources_EmployeePayHistory.csv - HumanResources - EmployeePayHistory
filepath: C:/data/data_files/tmp/HumanResources_EmployeePayHistory.csv
HumanResources_EmployeePayHistory
Creating Iceberg table: local.HumanResources.EmployeePayHistory
HumanResources_JobCandidate.csv - HumanResources - JobCandidate
filepath: C

In [ ]:
orc_output_path_ins = "C:/data/data_files/orc/" 

orc_dataframes={}
iceberg_dataframes = {}
LOCAL_ICEBERG_CATALOG = "local"

for df_name in dataframes:
    output_path = orc_output_path_ins + df_name[:df_name.find("_")] + "/" + df_name[df_name.find("_") + 1:]
    orc_df_name = df_name[df_name.find("_") + 1:]
    dataframes.get(df_name).write.mode("overwrite").orc(output_path)
    orc_df = spark.read.orc(output_path)
    orc_dataframes[orc_df_name]=orc_df
    
    # Initialize a dictionary to store the resulting Iceberg DataFrames

    # 1. Parse the names according to your convention:
    # df_name = "databaseName_tableName" (e.g., "mydb_mytable")
    
    # This will be the database name (e.g., 'mydb')
    db_name = df_name[:df_name.find("_")] 
    
    # This will be the table name (e.g., 'mytable')
    table_name = df_name[df_name.find("_") + 1:]
    
    # 2. Construct the Fully Qualified Iceberg Table Identifier
    # Iceberg requires a three-part name: catalog.database.table
    fq_iceberg_table = f"{LOCAL_ICEBERG_CATALOG}.{db_name}.{table_name}"
    
    print(f"Creating Iceberg table: {fq_iceberg_table}")

    # 3. Create the Iceberg table and write data using the DataFrameWriter API
    dataframes.get(df_name)\
        .write\
        .format("iceberg")\
        .mode("overwrite")\
        .saveAsTable(fq_iceberg_table)
    
    # 4. Read the data back from the new Iceberg table
    iceberg_df = spark.read.table(fq_iceberg_table)
    
    # 5. Store the resulting DataFrame for later use
    iceberg_dataframes[table_name] = iceberg_df
    
print("Iceberg table creation complete.")

    

In [ ]:
orc_output_path_ins = "C:/data/data_files/orc/" 

orc_dataframes={}
iceberg_dataframes = {}
LOCAL_ICEBERG_CATALOG = "local"

for df_name in dataframes:
    output_path = orc_output_path_ins + df_name[:df_name.find("_")] + "/" + df_name[df_name.find("_") + 1:]
    orc_df_name = df_name[df_name.find("_") + 1:]
    dataframes.get(df_name).write.mode("overwrite").orc(output_path)
    orc_df = spark.read.orc(output_path)
    orc_dataframes[orc_df_name]=orc_df
    
    # Initialize a dictionary to store the resulting Iceberg DataFrames

    # 1. Parse the names according to your convention:
    # df_name = "databaseName_tableName" (e.g., "mydb_mytable")
    
    # This will be the database name (e.g., 'mydb')
    db_name = df_name[:df_name.find("_")] 
    
    # This will be the table name (e.g., 'mytable')
    table_name = df_name[df_name.find("_") + 1:]
    
    # 2. Construct the Fully Qualified Iceberg Table Identifier
    # Iceberg requires a three-part name: catalog.database.table
    fq_iceberg_table = f"{LOCAL_ICEBERG_CATALOG}.{db_name}.{table_name}"
    
    print(f"Creating Iceberg table: {fq_iceberg_table}")

    # 3. Create the Iceberg table and write data using the DataFrameWriter API
    dataframes.get(df_name)\
        .write\
        .format("iceberg")\
        .mode("overwrite")\
        .saveAsTable(fq_iceberg_table)
    
    # 4. Read the data back from the new Iceberg table
    iceberg_df = spark.read.table(fq_iceberg_table)
    
    # 5. Store the resulting DataFrame for later use
    iceberg_dataframes[table_name] = iceberg_df
    
print("Iceberg table creation complete.")

    

In [ ]:
df_Department = spark.table("local.Person.Person")
# df_Department = spark.sql("Select * from local.HumanResources.Department")
df_Department.count()
# df_Department.show(10, truncate = False)

In [ ]:
p = spark.table("local.Person.Person").alias("p")
p.show(10, truncate = False)

In [ ]:
from pyspark.sql import functions as F

soh = orc_dataframes.get("SalesOrderHeader").alias("soh")
c = orc_dataframes.get("Customer").alias("c")
p = orc_dataframes.get("Person").alias("p")
sod = orc_dataframes.get("SalesOrderDetail").alias("sod")
prd = orc_dataframes.get("Product").alias("prd")

# soh = dataframes.get("Sales_SalesOrderHeader").alias("soh")
# c = dataframes.get("Sales_Customer").alias("c")
# p = dataframes.get("Person_Person").alias("p")
# sod = dataframes.get("Sales_SalesOrderDetail").alias("sod")
# prd = dataframes.get("Production_Product").alias("prd")
# prd.printSchema()
# sod.printSchema()
# soh.printSchema()
c.printSchema()

joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy"))
)

# joined_df.show(5, truncate=False)
mayFilter = joined_df.filter((F.year(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 5) & (F.col("AccountNumber") == "AW00029825"))

mayFilter.show(5, truncate=False)


In [ ]:
from pyspark.sql import SparkSession
# spark.conf.set("spark.sql.caseSensitive", "false")
# SparkSession.builder.config("spark.driver.memory", "4g")

# result_df = spark.sql(""" EXPLAIN 
# Select
# soh.SalesOrderID,
# soh.OrderDate,
# soh.DueDate,
# soh.ShipDate,
# soh.Status,
# soh.OnlineOrderFlag,
# soh.SalesOrderNumber,
# soh.PurchaseOrderNumber,
# soh.SubTotal,
# soh.TaxAmt,
# soh.Freight,
# soh.TotalDue,
# soh.Comment,
# cust.CustomerID,
# p.firstName,
# p.lastName,
# cust.AccountNumber,
# prd.name as Prod_Name,
# sod.OrderQty,
# sod.UnitPrice,
# sod.LineTotal
# FROM    local.Sales.SalesOrderHeader soh   INNER JOIN  
#         local.Sales.Customer cust
#             ON  soh.CustomerID = cust.CustomerID INNER JOIN
#         local.Person.Person p
#             ON  p.BusinessEntityID = cust.BusinessEntityID INNER JOIN
#         local.Sales.SalesOrderDetail sod
#             ON   soh.SalesOrderID = sod.SalesOrderID  INNER JOIN
#         local.Production.Product prd
#             ON sod.ProductID = prd.ProductID
# LIMIT 10
# """)

result_df = spark.sql("""
SELECT soh.SalesOrderID, cust.CustomerID
FROM local.Sales.SalesOrderHeader soh
JOIN local.Sales.Customer cust ON soh.CustomerID = cust.CustomerID
LIMIT 10
""")

# result_df = spark.sql("Select * from local.Person.Person")

# result_df.count()

# Print the schema
# result_df.printSchema()
result_df.show(10, truncate=False)

# spark.stop()


In [16]:
# Read the table directly using Iceberg catalog
df = spark.read.table("local.Sales.SalesOrderHeader")
df.show(5)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
df_soh = spark.read.table("local.Sales.SalesOrderHeader")
df_cust = spark.read.table("local.Sales.Customer")

df_joined = df_soh.join(df_cust, on="CustomerID", how="inner")
df_joined.select("SalesOrderID", "CustomerID").show(10)

In [15]:
df_soh = spark.read.table("local.Sales.SalesOrderHeader").cache()
df_cust = spark.read.table("local.Sales.Customer").cache()

df_joined = df_soh.join(df_cust, on="CustomerID", how="inner")
df_joined.select("SalesOrderID", "CustomerID").show(10)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [14]:
df_soh = spark.read.table("local.Sales.SalesOrderHeader")
df_cust = spark.read.table("local.Sales.Customer")

df_soh.createOrReplaceTempView("soh")
df_cust.createOrReplaceTempView("cust")

spark.sql("""
SELECT soh.SalesOrderID, cust.CustomerID
FROM soh
JOIN cust ON soh.CustomerID = cust.CustomerID
""").show()

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\data\python\Python312\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\data\python\Python312\Lib\socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\data\python\Python312\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\data\python\Python312\Lib\site-packages\py4j\clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while s

Py4JError: An error occurred while calling o282.sql